# SUBLIME — Kaggle Reproduction
**Paper:** [Towards Unsupervised Deep Graph Structure Learning (WWW 2022)](https://arxiv.org/pdf/2201.06367)

This notebook clones the repo, installs dependencies, and runs the SUBLIME experiments.
Use **T4 x2** or **P100** accelerator in Kaggle settings.

## 1. Install dependencies

In [ ]:
import subprocess, sys

# Check CUDA version to install the right DGL build
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout or result.stderr)

import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU count:', torch.cuda.device_count())

In [ ]:
!pip install dgl munkres networkx scikit-learn scipy --quiet


In [ ]:
import os, sys

def _setup_torchdata_stub(base='/tmp/_td_stub'):
    """Write real stub .py files for torchdata so DGL 2.x imports cleanly."""
    os.makedirs(f'{base}/torchdata/datapipes', exist_ok=True)
    os.makedirs(f'{base}/torchdata/dataloader2', exist_ok=True)
    _iter_src = (
        'try:\n'
        '    import torch.utils.data as _t\n'
        '    _B = _t.IterDataPipe\n'
        'except Exception:\n'
        '    _B = object\n'
        'class IterDataPipe(_B): pass\n'
        'class Mapper(IterDataPipe): pass\n'
        'class Filter(IterDataPipe): pass\n'
        'class Shuffler(IterDataPipe): pass\n'
        'class Collator(IterDataPipe): pass\n'
    )
    _graph_src = (
        'def traverse_dps(dp, *a, **kw): return {}\n'
        'def list_dps(*a, **kw): return []\n'
        'def find_dps(*a, **kw): return []\n'
        'def replace_dp(graph, old, new): return old\n'
    )
    files = {
        f'{base}/torchdata/__init__.py': '',
        f'{base}/torchdata/datapipes/__init__.py': '',
        f'{base}/torchdata/datapipes/iter.py': _iter_src,
        f'{base}/torchdata/datapipes/map.py': 'class MapDataPipe: pass\n',
        f'{base}/torchdata/dataloader2/__init__.py': '',
        f'{base}/torchdata/dataloader2/graph.py': _graph_src,
    }
    for path, src in files.items():
        with open(path, 'w') as fh:
            fh.write(src)
    for k in list(sys.modules):
        if k == 'torchdata' or k.startswith('torchdata.'):
            del sys.modules[k]
    if base not in sys.path:
        sys.path.insert(0, base)

_setup_torchdata_stub()

import dgl
print('DGL version:', dgl.__version__)


## 2. Clone the repo

In [ ]:
import os

REPO_DIR = '/kaggle/working/SUBLIME'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/wishaalk/SUBLIME.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls

## 3. Verify data files

In [ ]:
!ls data/

## 4. Quick smoke-test (1 trial, reduced epochs)

This uses **Cora / structure inference** — the first experiment from Table 2 of the paper.  
We set `-ntrials 1` and `-epochs 200` just to confirm everything runs end-to-end before launching the full experiment.

In [ ]:
!python main.py \
    -dataset cora \
    -ntrials 1 \
    -sparse 0 \
    -epochs_cls 200 \
    -lr_cls 0.001 \
    -w_decay_cls 0.0005 \
    -hidden_dim_cls 32 \
    -dropout_cls 0.5 \
    -dropedge_cls 0.25 \
    -nlayers_cls 2 \
    -patience_cls 10 \
    -epochs 200 \
    -lr 0.01 \
    -w_decay 0.0 \
    -hidden_dim 512 \
    -rep_dim 256 \
    -proj_dim 256 \
    -dropout 0.5 \
    -dropedge_rate 0.5 \
    -nlayers 2 \
    -type_learner fgp \
    -k 30 \
    -sim_function cosine \
    -activation_learner relu \
    -gsl_mode structure_inference \
    -eval_freq 20 \
    -tau 1 \
    -maskfeat_rate_learner 0.5 \
    -maskfeat_rate_anchor 0.7 \
    -contrast_batch_size 0 \
    -c 0 \
    -gpu 0

---
## 5. Full experiments (paper settings)

Run these cells one at a time. Each maps to a row in the paper's tables.

Expected results from the paper (Table 2 / Table 3):

| Dataset | Mode | Task | Reported Acc |
|---------|------|------|--------------|
| Cora | Structure Inference | Classification | 83.6% |
| Cora | Structure Refinement | Classification | 84.7% |
| Cora | Structure Refinement | Clustering (ACC) | 79.8% |
| Citeseer | Structure Inference | Classification | 73.6% |
| Citeseer | Structure Refinement | Classification | 73.9% |
| Citeseer | Structure Refinement | Clustering (ACC) | 69.6% |
| Pubmed | Structure Inference | Classification | 79.9% |
| Pubmed | Structure Refinement | Classification | 81.3% |

### 5a. Cora — Node Classification @ Structure Inference

In [ ]:
!python main.py \
    -dataset cora -ntrials 5 -sparse 0 \
    -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.0005 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.25 -nlayers_cls 2 -patience_cls 10 \
    -epochs 4000 -lr 0.01 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner fgp -k 30 -sim_function cosine -activation_learner relu \
    -gsl_mode structure_inference -eval_freq 20 -tau 1 \
    -maskfeat_rate_learner 0.5 -maskfeat_rate_anchor 0.7 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5b. Cora — Node Classification @ Structure Refinement

In [ ]:
!python main.py \
    -dataset cora -ntrials 5 -sparse 0 \
    -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.0005 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.75 -nlayers_cls 2 -patience_cls 10 \
    -epochs 4000 -lr 0.01 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner fgp -k 30 -sim_function cosine -activation_learner relu \
    -gsl_mode structure_refinement -eval_freq 50 -tau 0.9999 \
    -maskfeat_rate_learner 0.7 -maskfeat_rate_anchor 0.6 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5c. Cora — Node Clustering @ Structure Refinement

In [ ]:
!python main.py \
    -dataset cora -downstream_task clustering -ntrials 10 -sparse 0 \
    -epochs 2500 -lr 0.001 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner fgp -k 20 -sim_function cosine -activation_learner relu \
    -gsl_mode structure_refinement -eval_freq 100 -tau 0.9999 \
    -maskfeat_rate_learner 0.1 -maskfeat_rate_anchor 0.8 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5d. Citeseer — Node Classification @ Structure Inference

In [ ]:
!python main.py \
    -dataset citeseer -ntrials 5 -sparse 0 \
    -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.05 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.5 -nlayers_cls 2 -patience_cls 10 \
    -epochs 1000 -lr 0.01 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.25 -nlayers 2 \
    -type_learner att -k 20 -sim_function cosine -activation_learner tanh \
    -gsl_mode structure_inference -eval_freq 50 -tau 0.9999 \
    -maskfeat_rate_learner 0.8 -maskfeat_rate_anchor 0.7 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5e. Citeseer — Node Classification @ Structure Refinement

In [ ]:
!python main.py \
    -dataset citeseer -ntrials 5 -sparse 0 \
    -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.05 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.5 -nlayers_cls 2 -patience_cls 10 \
    -epochs 1000 -lr 0.001 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.25 -nlayers 2 \
    -type_learner att -k 20 -sim_function cosine -activation_learner tanh \
    -gsl_mode structure_refinement -eval_freq 20 -tau 0.9999 \
    -maskfeat_rate_learner 0.6 -maskfeat_rate_anchor 0.8 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5f. Citeseer — Node Clustering @ Structure Refinement

In [ ]:
!python main.py \
    -dataset citeseer -downstream_task clustering -ntrials 10 -sparse 0 \
    -epochs 1000 -lr 0.001 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner att -k 20 -sim_function cosine -activation_learner tanh \
    -gsl_mode structure_refinement -eval_freq 100 -tau 0.999 \
    -maskfeat_rate_learner 0.4 -maskfeat_rate_anchor 0.9 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5g. Pubmed — Node Classification @ Structure Inference

In [ ]:
!python main.py \
    -dataset pubmed -ntrials 5 -sparse 1 \
    -epochs_cls 200 -lr_cls 0.01 -w_decay_cls 0.0005 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.25 -nlayers_cls 2 -patience_cls 10 \
    -epochs 2000 -lr 0.01 -w_decay 0.0 -hidden_dim 128 -rep_dim 64 -proj_dim 64 \
    -dropout 0.5 -dropedge_rate 0.25 -nlayers 2 \
    -type_learner att -k 15 -sim_function cosine -activation_learner tanh \
    -gsl_mode structure_inference -eval_freq 20 -tau 1 \
    -maskfeat_rate_learner 0.8 -maskfeat_rate_anchor 0.3 \
    -contrast_batch_size 2000 -c 0 -gpu 0

### 5h. Pubmed — Node Classification @ Structure Refinement

In [ ]:
!python main.py \
    -dataset pubmed -ntrials 5 -sparse 1 \
    -epochs_cls 200 -lr_cls 0.01 -w_decay_cls 0.0005 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.25 -nlayers_cls 2 -patience_cls 10 \
    -epochs 1500 -lr 0.001 -w_decay 0.0 -hidden_dim 128 -rep_dim 64 -proj_dim 64 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner mlp -k 10 -sim_function cosine -activation_learner relu \
    -gsl_mode structure_refinement -eval_freq 20 -tau 0.999 \
    -maskfeat_rate_learner 0.4 -maskfeat_rate_anchor 0.4 \
    -contrast_batch_size 2000 -c 50 -gpu 0